# 12. 参数高效微调与推理解码体系

学完模型结构之后，下一个现实问题是：

- 模型太大，怎样低成本微调？
- 推理阶段，怎样控制输出质量、多样性和稳定性？

这就是 PEFT（参数高效微调）和 decoding（解码策略）要解决的问题。

## 参数高效微调与推理解码的形式化定义

参数高效微调（PEFT）是一类仅更新少量附加参数而非全量底座参数的适配方法，目标是在显著降低显存和计算成本的同时保留大模型迁移能力。推理解码则是在模型给出下一 token 概率分布后，对候选 token 进行搜索或采样的策略集合。

## 输入、输出与参数化方式

在 LoRA 中，输入仍经过原始线性层，但额外叠加一个低秩增量分支；在推理解码中，输入是模型生成的 logits 或概率分布，输出是下一个 token 的选取结果及整个序列展开路径。

## 结构分解与信息流

            PEFT 关注的是“如何改模型”；解码关注的是“如何使用模型输出分布”。

LoRA 的结构是在冻结的权重矩阵旁引入 $BA$ 低秩分支；QLoRA 在此基础上进一步压缩底座参数表示。解码策略则包括 greedy、beam search、temperature sampling、top-k 与 top-p 等，它们并不改变模型参数，只改变输出路径搜索方式。

## 优化目标与训练机制

            PEFT 训练目标与原任务目标保持一致，区别仅在于可训练参数集合更小。LoRA 通过低秩约束假设“任务适配所需的权重更新位于一个低维子空间中”。

解码阶段的目标则是平衡：

- 确定性
- 多样性
- 流畅性
- 可控性

## 计算复杂度、统计性质与工程代价

            LoRA 的参数量与秩 $r$ 成正比，远小于全量微调；QLoRA 进一步降低显存开销，因此成为消费级 GPU 上微调大模型的重要路线。

解码复杂度方面，greedy 最便宜，beam search 更昂贵，采样类方法则在质量与多样性间做折中。

## 与相邻模型的差异

            与全量微调相比，PEFT 更经济、更灵活，适合多任务并存。
与 prompt engineering 相比，PEFT 能真正改变参数。
在解码层面，beam search 更偏搜索最优，top-p / temperature 更偏自然生成。它们不是互相替代关系，而是面向不同输出目标的配置工具。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

ax.add_patch(plt.Rectangle((1.0, 1.9), 1.3, 1.2, facecolor="#4C78A8", edgecolor="black"))
ax.text(1.65, 2.5, "输入 x", ha="center", va="center", fontsize=11)

ax.add_patch(plt.Rectangle((4.0, 3.0), 1.8, 1.0, facecolor="#F58518", edgecolor="black"))
ax.text(4.9, 3.5, "冻结主权重 W", ha="center", va="center", fontsize=11)

ax.add_patch(plt.Rectangle((4.0, 1.1), 1.1, 0.8, facecolor="#E45756", edgecolor="black"))
ax.text(4.55, 1.5, "A", ha="center", va="center", fontsize=11)
ax.add_patch(plt.Rectangle((6.0, 1.1), 1.1, 0.8, facecolor="#72B7B2", edgecolor="black"))
ax.text(6.55, 1.5, "B", ha="center", va="center", fontsize=11)
ax.text(5.25, 0.55, "低秩增量 ΔW = BA", ha="center", fontsize=11)

ax.add_patch(plt.Circle((8.4, 2.5), 0.35, color="#54A24B"))
ax.text(8.4, 2.5, "+", ha="center", va="center", fontsize=16, color="white")

ax.add_patch(plt.Rectangle((10.0, 1.9), 1.3, 1.2, facecolor="#B279A2", edgecolor="black"))
ax.text(10.65, 2.5, "输出 y", ha="center", va="center", fontsize=11)

ax.annotate("", xy=(4.0, 3.5), xytext=(2.3, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(4.0, 1.5), xytext=(2.3, 2.5), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(6.0, 1.5), xytext=(5.1, 1.5), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(8.05, 2.7), xytext=(5.8, 3.4), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(8.05, 2.3), xytext=(7.1, 1.5), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(10.0, 2.5), xytext=(8.75, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("LoRA 的低秩增量结构")
plt.show()

## 先建立直觉

            大模型真正落地时，最现实的问题不是“它能不能回答”，而是：

- 我有没有足够算力去训练或微调它？
- 我能不能控制它输出得更稳定、更自然？

参数高效微调解决的是“怎么少花资源改模型”；
解码策略解决的是“同一个模型，怎么让它按你想要的风格输出”。

## 架构是怎么工作的

            LoRA 不是换掉原模型，而是在原有线性层旁边加一个很小的低秩分支。

这意味着：

- 原模型大部分参数冻结
- 只训练很少一部分增量参数
- 推理时再把增量作用叠加回去

解码策略则发生在模型输出概率之后：

- 模型先给每个 token 一个概率
- 你再决定怎么从这些概率里选下一个 token

## 训练时到底发生了什么

            PEFT 训练时，真正更新的参数很少，因此：

- 显存需求更低
- 训练更快
- 更适合多任务、多 LoRA 并存

解码不属于训练，但它极大影响用户看到的效果。
同一个模型，使用 Greedy、Top-p、Temperature=0.7、Beam Search，输出风格会明显不同。

## 什么时候该用它

            如果你想做业务定制：

- 数据量不大：优先 LoRA
- 显存更紧：优先 QLoRA
- 任务偏格式化：解码可偏保守
- 任务偏创作：解码可更随机

## 最常见的误区

            常见误区：

1. **误以为 LoRA 可以替代所有全量微调**
   对大多数任务够用，但不是所有场景都能完全替代。

2. **误以为 Temperature 越高越“聪明”**
   它只是更随机，不是更聪明。

3. **误以为 Beam Search 一定比采样好**
   开放式生成里，Beam Search 反而可能更僵硬。

## 1. LoRA 的核心思想

假设原始权重矩阵为 $W \in \mathbb{R}^{d_{out}\times d_{in}}$。

LoRA 不直接训练整个 $W$，而是只训练一个低秩增量：

$$
\Delta W = BA
$$

其中：

- $A \in \mathbb{R}^{r \times d_{in}}$
- $B \in \mathbb{R}^{d_{out} \times r}$
- $r \ll \min(d_{in}, d_{out})$

最终权重：

$$
W' = W + \frac{\alpha}{r} BA
$$

这样可以显著减少可训练参数数量。

## 2. 常见参数高效微调方法

- `LoRA`：最常见，工程生态最好
- `QLoRA`：量化底座模型，再训练低秩增量，显著节省显存
- `Prompt Tuning`：只学习少量软提示向量
- `Prefix Tuning`：为注意力层增加可学习前缀
- `Adapter`：在主干网络中插入小型瓶颈模块

一般来说：

- 资源有限时，优先 LoRA / QLoRA
- 需要最小侵入时，可考虑 Prompt Tuning
- 需要更强任务适配时，LoRA 往往更实用

## 3. 推理解码策略

### Greedy Decoding

每一步直接选概率最大的 token：

$$
x_t = \arg\max_i p(i \mid x_{<t})
$$

### Temperature Sampling

$$
p_i' = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}
$$

- $T < 1$：更保守
- $T > 1$：更随机

### Top-k

只在概率最高的 $k$ 个 token 中采样。

### Top-p / Nucleus Sampling

选择累计概率达到阈值 $p$ 的最小 token 集合，再在其中采样。

### Beam Search

保留多个候选序列，适合更偏“搜索”的任务，但在开放式生成中未必自然。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
# 低秩近似实验：观察 rank 越低，能否用更少参数逼近原矩阵。
torch.manual_seed(42)
W = torch.randn(256, 256)
U, S, Vh = torch.linalg.svd(W, full_matrices=False)

rank_records = []
total_params = W.numel()
for rank in [2, 4, 8, 16, 32, 64, 128]:
    W_approx = (U[:, :rank] * S[:rank]) @ Vh[:rank, :]
    rel_error = torch.norm(W - W_approx) / torch.norm(W)
    lora_params = rank * (256 + 256)
    rank_records.append(
        {
            "rank": rank,
            "相对重建误差": rel_error.item(),
            "LoRA 参数量": lora_params,
            "参数占原矩阵比例": lora_params / total_params,
        }
    )

rank_df = pd.DataFrame(rank_records)
rank_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(rank_df["rank"], rank_df["相对重建误差"], marker="o")
axes[0].set_title("低秩近似的 rank 与重建误差")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("relative reconstruction error")

axes[1].plot(rank_df["rank"], rank_df["参数占原矩阵比例"], marker="o", color="darkgreen")
axes[1].set_title("低秩近似的 rank 与参数占比")
axes[1].set_xlabel("rank")
axes[1].set_ylabel("parameter ratio")

plt.tight_layout()
plt.show()

In [ ]:
# 用一个玩具 token 概率分布展示不同解码策略的差异。
vocab = ["A", "B", "C", "D", "E", "F", "G", "H"]
logits = torch.tensor([4.2, 3.9, 2.8, 2.3, 1.6, 1.1, 0.2, -0.5], dtype=torch.float32)


def softmax_with_temperature(logits, temperature=1.0):
    scaled = logits / temperature
    return torch.softmax(scaled, dim=-1)


def top_k_sample(probs, k=3):
    values, indices = torch.topk(probs, k=k)
    sampled_local = torch.multinomial(values / values.sum(), num_samples=1).item()
    return indices[sampled_local].item()


def top_p_sample(probs, p=0.8):
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=0)
    keep_mask = cumulative <= p
    keep_mask[0] = True
    kept_probs = sorted_probs[keep_mask]
    kept_indices = sorted_indices[keep_mask]
    sampled_local = torch.multinomial(kept_probs / kept_probs.sum(), num_samples=1).item()
    return kept_indices[sampled_local].item()


probs_t1 = softmax_with_temperature(logits, temperature=1.0)
probs_t07 = softmax_with_temperature(logits, temperature=0.7)
probs_t13 = softmax_with_temperature(logits, temperature=1.3)

print("Greedy:", vocab[int(torch.argmax(probs_t1))])
print("Top-k sample:", vocab[top_k_sample(probs_t1, k=3)])
print("Top-p sample:", vocab[top_p_sample(probs_t1, p=0.8)])

In [ ]:
decode_df = pd.DataFrame(
    {
        "token": vocab,
        "T=0.7": probs_t07.numpy(),
        "T=1.0": probs_t1.numpy(),
        "T=1.3": probs_t13.numpy(),
    }
)

decode_long = decode_df.melt(id_vars="token", var_name="temperature", value_name="prob")

plt.figure(figsize=(12, 6))
sns.barplot(data=decode_long, x="token", y="prob", hue="temperature")
plt.title("不同 temperature 下的 token 概率分布")
plt.show()

In [ ]:
# 用简单的序列得分说明 Beam Search 与 Greedy 的差异。
# 这里只做一个手工构造的树搜索示意，不依赖大模型。
transitions = {
    "<BOS>": {"我": 0.55, "今天": 0.45},
    "我": {"喜欢": 0.60, "想": 0.40},
    "今天": {"天气": 0.70, "想": 0.30},
    "喜欢": {"学习": 0.65, "跑步": 0.35},
    "想": {"学习": 0.52, "休息": 0.48},
    "天气": {"很好": 0.90, "一般": 0.10},
}

def greedy_path():
    path = ["<BOS>"]
    score = 0.0
    current = "<BOS>"
    while current in transitions:
        next_token, prob = max(transitions[current].items(), key=lambda x: x[1])
        path.append(next_token)
        score += math.log(prob)
        current = next_token
    return path[1:], score

def beam_search(beam_width=2, max_steps=3):
    beams = [(["<BOS>"], 0.0)]
    for _ in range(max_steps):
        new_beams = []
        for path, score in beams:
            current = path[-1]
            if current not in transitions:
                new_beams.append((path, score))
                continue
            for next_token, prob in transitions[current].items():
                new_beams.append((path + [next_token], score + math.log(prob)))
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
    return [(path[1:], score) for path, score in beams]

print("Greedy 结果:", greedy_path())
print("Beam Search 结果:", beam_search(beam_width=2, max_steps=3))

## 4. 使用建议

### 微调方法

- `LoRA`：绝大多数私有数据定制任务的第一选择
- `QLoRA`：显存紧张、只有消费级 GPU 时很有价值
- `全量微调`：只有在资源足够、任务偏移很大时才考虑

### 解码策略

- `Greedy`：稳定、可重复，适合分类、结构化抽取
- `Beam Search`：适合机器翻译、摘要等更偏搜索的任务
- `Top-k / Top-p + Temperature`：适合开放式生成和创作

实际部署中，很多高质量对话系统都会把：

- system prompt
- 采样参数
- repetition penalty
- stop tokens

一起当作“推理配置”的一部分来调优。